# Variações dos Datasets Base

**IMPORTANTE:** Execute primeiro o notebook `tratamento_datasets.ipynb` para gerar os datasets base.

Este notebook cria 3 variações dos datasets base:
1. **Janelas Deslizantes**: adiciona 12 colunas (2, 4, 6 dias no passado para Open, High, Low, Close)
2. **Variável Dummy**: adiciona variável dicotômica de período (0, 1, 2) com One-Hot Encoding
3. **Indicadores Técnicos**: adiciona OBV, FWMA, TEMA, HLC3 e Bollinger Bands

Cada variação gera 6 arquivos (3 empresas × 2 tipos: regressão e classificação).

In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 1. Configuração de pastas

In [3]:
BASE_DIR = Path('..').resolve()
DATASETS_DIR = BASE_DIR / 'datasets'

# Pastas de entrada (datasets base gerados pelo tratamento_datasets.ipynb)
INPUT_REGRESSAO_DIR = DATASETS_DIR / 'datasets_base' / 'regressao'
INPUT_CLASSIFICACAO_DIR = DATASETS_DIR / 'datasets_base' / 'classificacao'

# Pastas de saída para cada variação
OUTPUT_JANELAS_REG = DATASETS_DIR / 'datasets_janelas' / 'regressao'
OUTPUT_JANELAS_CLF = DATASETS_DIR / 'datasets_janelas' / 'classificacao'

OUTPUT_DUMMY_REG = DATASETS_DIR / 'datasets_dummy' / 'regressao'
OUTPUT_DUMMY_CLF = DATASETS_DIR / 'datasets_dummy' / 'classificacao'

OUTPUT_INDICADORES_REG = DATASETS_DIR / 'datasets_indicadores' / 'regressao'
OUTPUT_INDICADORES_CLF = DATASETS_DIR / 'datasets_indicadores' / 'classificacao'

# Criar todas as pastas de saída
for folder in [OUTPUT_JANELAS_REG, OUTPUT_JANELAS_CLF, OUTPUT_DUMMY_REG, OUTPUT_DUMMY_CLF, OUTPUT_INDICADORES_REG, OUTPUT_INDICADORES_CLF]:
    folder.mkdir(parents=True, exist_ok=True)

print('Pastas de entrada:')
print('  - Regressão:', INPUT_REGRESSAO_DIR)
print('  - Classificação:', INPUT_CLASSIFICACAO_DIR)
print('\nPastas de saída criadas:')
print('  - Janelas Deslizantes:', OUTPUT_JANELAS_REG.parent)
print('  - Variável Dummy:', OUTPUT_DUMMY_REG.parent)
print('  - Indicadores Técnicos:', OUTPUT_INDICADORES_REG.parent)

Pastas de entrada:
  - Regressão: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_base/regressao
  - Classificação: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_base/classificacao

Pastas de saída criadas:
  - Janelas Deslizantes: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_janelas
  - Variável Dummy: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_dummy
  - Indicadores Técnicos: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_indicadores


In [4]:
# Verificar se os datasets base existem
csv_reg = sorted(INPUT_REGRESSAO_DIR.glob('*_tratado.csv'))
csv_clf = sorted(INPUT_CLASSIFICACAO_DIR.glob('*_tratado.csv'))

if not csv_reg or not csv_clf:
    raise FileNotFoundError(
        'Datasets base não encontrados! Execute primeiro o notebook tratamento_datasets.ipynb'
    )

print(f'Datasets base encontrados: {len(csv_reg)} para regressão, {len(csv_clf)} para classificação')
print('\nArquivos de regressão:')
for f in csv_reg:
    print('  -', f.name)
print('\nArquivos de classificação:')
for f in csv_clf:
    print('  -', f.name)

Datasets base encontrados: 3 para regressão, 3 para classificação

Arquivos de regressão:
  - AGRO3_tratado.csv
  - SLCE3_tratado.csv
  - SOJA3_tratado.csv

Arquivos de classificação:
  - AGRO3_tratado.csv
  - SLCE3_tratado.csv
  - SOJA3_tratado.csv


## 2. Funções de transformação

### 2.1 Janelas Deslizantes (Lag Features)

In [5]:
def adicionar_janelas_deslizantes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona janelas deslizantes para trás (2, 4 e 6 dias) para Open, High, Low e Close.
    Gera 12 novas colunas (4 features × 3 períodos).
    """
    df = df.copy()
    features = ['Open', 'High', 'Low', 'Close']
    lags = [2, 4, 6]
    
    for feature in features:
        if feature in df.columns:
            for lag in lags:
                col_name = f'{feature}_lag{lag}d'
                df[col_name] = df[feature].shift(lag)
    
    # Remove linhas com NaN geradas pelos shifts
    df = df.dropna()
    
    return df

print('Função adicionar_janelas_deslizantes() definida.')

Função adicionar_janelas_deslizantes() definida.


### 2.2 Variável Dummy de Período

In [6]:
def adicionar_variavel_dummy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona variável dummy de período baseada em datas específicas.
    - 0: antes de 2021-05-05 (período inicial)
    - 1: entre 2021-05-05 e 2022-02-07 (período de alta volatilidade)
    - 2: após 2022-02-07 (período de estabilização)
    
    Aplica One-Hot Encoding (drop='first') gerando period_1 e period_2.
    """
    df = df.copy()
    
    # Garantir que o índice é datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date')
        else:
            df.index = pd.to_datetime(df.index)
    
    # Criar variável dummy categórica
    df['dummy_period'] = 0
    df.loc[df.index >= pd.to_datetime('2021-05-05'), 'dummy_period'] = 1
    df.loc[df.index >= pd.to_datetime('2022-02-07'), 'dummy_period'] = 2
    
    # Aplicar One-Hot Encoding (drop='first' para evitar multicolinearidade)
    dummy_encoded = pd.get_dummies(df['dummy_period'], prefix='period', drop_first=True, dtype=int)
    
    # Concatenar com DataFrame original e remover coluna dummy_period original
    df = pd.concat([df, dummy_encoded], axis=1)
    df = df.drop(columns=['dummy_period'])
    
    return df

print('Função adicionar_variavel_dummy() definida.')

Função adicionar_variavel_dummy() definida.


### 2.3 Indicadores Técnicos

In [7]:
def calcular_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """
    On-Balance Volume (OBV): indicador de momentum baseado em volume.
    """
    direction = np.where(close > close.shift(1), 1, np.where(close < close.shift(1), -1, 0))
    obv = (direction * volume).cumsum()
    return pd.Series(obv, index=close.index, name='OBV')


def calcular_fwma(series: pd.Series, period: int = 14) -> pd.Series:
    """
    Fibonacci Weighted Moving Average (FWMA).
    Usa pesos baseados na sequência de Fibonacci.
    """
    # Gerar pesos de Fibonacci
    fib = [1, 1]
    for i in range(2, period):
        fib.append(fib[i-1] + fib[i-2])
    weights = np.array(fib[:period], dtype=float)
    weights = weights / weights.sum()
    
    # Calcular média ponderada
    fwma = series.rolling(window=period).apply(lambda x: np.dot(x, weights), raw=True)
    return pd.Series(fwma, index=series.index, name='FWMA')


def calcular_tema(series: pd.Series, period: int = 14) -> pd.Series:
    """
    Triple Exponential Moving Average (TEMA).
    TEMA = 3*EMA1 - 3*EMA2 + EMA3
    """
    ema1 = series.ewm(span=period, adjust=False).mean()
    ema2 = ema1.ewm(span=period, adjust=False).mean()
    ema3 = ema2.ewm(span=period, adjust=False).mean()
    tema = 3 * ema1 - 3 * ema2 + ema3
    return pd.Series(tema, index=series.index, name='TEMA')


def calcular_hlc3(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    """
    HLC3 (Typical Price): média de High, Low e Close.
    """
    hlc3 = (high + low + close) / 3
    return pd.Series(hlc3, index=high.index, name='HLC3')


def calcular_bollinger_bands(series: pd.Series, period: int = 20, std_dev: float = 2.0) -> pd.DataFrame:
    """
    Bollinger Bands: banda superior, média móvel e banda inferior.
    """
    sma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    
    upper_band = sma + (std_dev * std)
    lower_band = sma - (std_dev * std)
    
    return pd.DataFrame({
        'BB_upper': upper_band,
        'BB_middle': sma,
        'BB_lower': lower_band
    }, index=series.index)


print('Funções de indicadores técnicos definidas:')
print('  - calcular_obv()')
print('  - calcular_fwma()')
print('  - calcular_tema()')
print('  - calcular_hlc3()')
print('  - calcular_bollinger_bands()')

Funções de indicadores técnicos definidas:
  - calcular_obv()
  - calcular_fwma()
  - calcular_tema()
  - calcular_hlc3()
  - calcular_bollinger_bands()


In [8]:
def adicionar_indicadores_tecnicos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona todos os indicadores técnicos ao DataFrame:
    - OBV (On-Balance Volume)
    - FWMA (Fibonacci Weighted Moving Average)
    - TEMA (Triple Exponential Moving Average)
    - HLC3 (Typical Price)
    - Bollinger Bands (upper, middle, lower)
    """
    df = df.copy()
    
    # Verificar colunas necessárias
    required = ['Open', 'High', 'Low', 'Close', 'Volume']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Colunas ausentes para indicadores técnicos: {missing}')
    
    # OBV
    df['OBV'] = calcular_obv(df['Close'], df['Volume'])
    
    # FWMA (usando Close)
    df['FWMA'] = calcular_fwma(df['Close'], period=14)
    
    # TEMA (usando Close)
    df['TEMA'] = calcular_tema(df['Close'], period=14)
    
    # HLC3
    df['HLC3'] = calcular_hlc3(df['High'], df['Low'], df['Close'])
    
    # Bollinger Bands
    bb = calcular_bollinger_bands(df['Close'], period=20, std_dev=2.0)
    df = pd.concat([df, bb], axis=1)
    
    # Remove linhas com NaN geradas pelos cálculos
    df = df.dropna()
    
    return df

print('Função adicionar_indicadores_tecnicos() definida.')

Função adicionar_indicadores_tecnicos() definida.


## 3. Processamento em lote

### 3.1 Datasets com Janelas Deslizantes

In [9]:
print('=' * 80)
print('VARIAÇÃO 1: JANELAS DESLIZANTES (LAG FEATURES)')
print('=' * 80)

# Processar arquivos de regressão
print('\n--- Processando datasets de REGRESSÃO ---')
for file_path in csv_reg:
    print(f'\nProcessando: {file_path.name}')
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'  Shape original: {df.shape}')
    
    df_janelas = adicionar_janelas_deslizantes(df)
    print(f'  Shape com janelas: {df_janelas.shape}')
    
    output_path = OUTPUT_JANELAS_REG / file_path.name
    df_janelas.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

# Processar arquivos de classificação
print('\n--- Processando datasets de CLASSIFICAÇÃO ---')
for file_path in csv_clf:
    print(f'\nProcessando: {file_path.name}')
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'  Shape original: {df.shape}')
    
    df_janelas = adicionar_janelas_deslizantes(df)
    print(f'  Shape com janelas: {df_janelas.shape}')
    
    output_path = OUTPUT_JANELAS_CLF / file_path.name
    df_janelas.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

print('\n✓ Datasets com janelas deslizantes gerados com sucesso!')

VARIAÇÃO 1: JANELAS DESLIZANTES (LAG FEATURES)

--- Processando datasets de REGRESSÃO ---

Processando: AGRO3_tratado.csv
  Shape original: (1708, 9)
  Shape com janelas: (1702, 21)


  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_janelas/regressao/AGRO3_tratado.csv

Processando: SLCE3_tratado.csv
  Shape original: (1708, 9)
  Shape com janelas: (1702, 21)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_janelas/regressao/SLCE3_tratado.csv

Processando: SOJA3_tratado.csv
  Shape original: (887, 9)
  Shape com janelas: (881, 21)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_janelas/regressao/SOJA3_tratado.csv

--- Processando datasets de CLASSIFICAÇÃO ---

Processando: AGRO3_tratado.csv
  Shape original: (1708, 9)
  Shape com janelas: (1702, 21)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_janelas/classificacao/AGRO3_tratado.csv

Processando: S

### 3.2 Datasets com Variável Dummy

In [10]:
print('=' * 80)
print('VARIAÇÃO 2: VARIÁVEL DUMMY DE PERÍODO')
print('=' * 80)

# Processar arquivos de regressão
print('\n--- Processando datasets de REGRESSÃO ---')
for file_path in csv_reg:
    print(f'\nProcessando: {file_path.name}')
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'  Shape original: {df.shape}')
    
    df_dummy = adicionar_variavel_dummy(df)
    print(f'  Shape com dummy: {df_dummy.shape}')
    
    output_path = OUTPUT_DUMMY_REG / file_path.name
    df_dummy.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

# Processar arquivos de classificação
print('\n--- Processando datasets de CLASSIFICAÇÃO ---')
for file_path in csv_clf:
    print(f'\nProcessando: {file_path.name}')
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'  Shape original: {df.shape}')
    
    df_dummy = adicionar_variavel_dummy(df)
    print(f'  Shape com dummy: {df_dummy.shape}')
    
    output_path = OUTPUT_DUMMY_CLF / file_path.name
    df_dummy.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

print('\n✓ Datasets com variável dummy gerados com sucesso!')

VARIAÇÃO 2: VARIÁVEL DUMMY DE PERÍODO

--- Processando datasets de REGRESSÃO ---

Processando: AGRO3_tratado.csv
  Shape original: (1708, 9)
  Shape com dummy: (1708, 11)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_dummy/regressao/AGRO3_tratado.csv

Processando: SLCE3_tratado.csv
  Shape original: (1708, 9)
  Shape com dummy: (1708, 11)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_dummy/regressao/SLCE3_tratado.csv

Processando: SOJA3_tratado.csv
  Shape original: (887, 9)
  Shape com dummy: (887, 11)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_dummy/regressao/SOJA3_tratado.csv

--- Processando datasets de CLASSIFICAÇÃO ---

Processando: AGRO3_tratado.csv
  Shape original: (1708, 9)
  Shape com dummy: (1708, 11)
  Salvo em: /home/gabriel

### 3.3 Datasets com Indicadores Técnicos

In [11]:
print('=' * 80)
print('VARIAÇÃO 3: INDICADORES TÉCNICOS')
print('=' * 80)

# Processar arquivos de regressão
print('\n--- Processando datasets de REGRESSÃO ---')
for file_path in csv_reg:
    print(f'\nProcessando: {file_path.name}')
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'  Shape original: {df.shape}')
    
    df_indicadores = adicionar_indicadores_tecnicos(df)
    print(f'  Shape com indicadores: {df_indicadores.shape}')
    
    output_path = OUTPUT_INDICADORES_REG / file_path.name
    df_indicadores.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

# Processar arquivos de classificação
print('\n--- Processando datasets de CLASSIFICAÇÃO ---')
for file_path in csv_clf:
    print(f'\nProcessando: {file_path.name}')
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'  Shape original: {df.shape}')
    
    df_indicadores = adicionar_indicadores_tecnicos(df)
    print(f'  Shape com indicadores: {df_indicadores.shape}')
    
    output_path = OUTPUT_INDICADORES_CLF / file_path.name
    df_indicadores.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

print('\n✓ Datasets com indicadores técnicos gerados com sucesso!')

VARIAÇÃO 3: INDICADORES TÉCNICOS

--- Processando datasets de REGRESSÃO ---

Processando: AGRO3_tratado.csv
  Shape original: (1708, 9)
  Shape com indicadores: (1689, 16)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_indicadores/regressao/AGRO3_tratado.csv

Processando: SLCE3_tratado.csv
  Shape original: (1708, 9)
  Shape com indicadores: (1689, 16)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_indicadores/regressao/SLCE3_tratado.csv

Processando: SOJA3_tratado.csv
  Shape original: (887, 9)
  Shape com indicadores: (868, 16)
  Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/datasets/datasets_indicadores/regressao/SOJA3_tratado.csv

--- Processando datasets de CLASSIFICAÇÃO ---

Processando: AGRO3_tratado.csv
  Shape original: (1708, 9)
  Shape com indicadores:

## 4. Resumo final

In [12]:
print('=' * 80)
print('RESUMO FINAL')
print('=' * 80)

# Contar arquivos gerados
janelas_reg = sorted(OUTPUT_JANELAS_REG.glob('*.csv'))
janelas_clf = sorted(OUTPUT_JANELAS_CLF.glob('*.csv'))
dummy_reg = sorted(OUTPUT_DUMMY_REG.glob('*.csv'))
dummy_clf = sorted(OUTPUT_DUMMY_CLF.glob('*.csv'))
indicadores_reg = sorted(OUTPUT_INDICADORES_REG.glob('*.csv'))
indicadores_clf = sorted(OUTPUT_INDICADORES_CLF.glob('*.csv'))

print('\n📁 JANELAS DESLIZANTES (datasets_janelas/):')
print(f'  Regressão: {len(janelas_reg)} arquivos')
for f in janelas_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(janelas_clf)} arquivos')
for f in janelas_clf:
    print(f'    - {f.name}')

print('\n📁 VARIÁVEL DUMMY (datasets_dummy/):')
print(f'  Regressão: {len(dummy_reg)} arquivos')
for f in dummy_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(dummy_clf)} arquivos')
for f in dummy_clf:
    print(f'    - {f.name}')

print('\n📁 INDICADORES TÉCNICOS (datasets_indicadores/):')
print(f'  Regressão: {len(indicadores_reg)} arquivos')
for f in indicadores_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(indicadores_clf)} arquivos')
for f in indicadores_clf:
    print(f'    - {f.name}')

total = len(janelas_reg) + len(janelas_clf) + len(dummy_reg) + len(dummy_clf) + len(indicadores_reg) + len(indicadores_clf)
print(f'\n✓ Total de {total} datasets gerados com sucesso!')

RESUMO FINAL

📁 JANELAS DESLIZANTES (datasets_janelas/):
  Regressão: 3 arquivos
    - AGRO3_tratado.csv
    - SLCE3_tratado.csv
    - SOJA3_tratado.csv
  Classificação: 3 arquivos
    - AGRO3_tratado.csv
    - SLCE3_tratado.csv
    - SOJA3_tratado.csv

📁 VARIÁVEL DUMMY (datasets_dummy/):
  Regressão: 3 arquivos
    - AGRO3_tratado.csv
    - SLCE3_tratado.csv
    - SOJA3_tratado.csv
  Classificação: 3 arquivos
    - AGRO3_tratado.csv
    - SLCE3_tratado.csv
    - SOJA3_tratado.csv

📁 INDICADORES TÉCNICOS (datasets_indicadores/):
  Regressão: 3 arquivos
    - AGRO3_tratado.csv
    - SLCE3_tratado.csv
    - SOJA3_tratado.csv
  Classificação: 3 arquivos
    - AGRO3_tratado.csv
    - SLCE3_tratado.csv
    - SOJA3_tratado.csv

✓ Total de 18 datasets gerados com sucesso!


## 5. Visualização das features adicionadas

In [13]:
# Carregar um exemplo de cada variação para mostrar as colunas
print('=' * 80)
print('COLUNAS ADICIONADAS EM CADA VARIAÇÃO')
print('=' * 80)

# Exemplo: primeiro arquivo de regressão
if janelas_reg:
    df_exemplo = pd.read_csv(janelas_reg[0], index_col=0, nrows=5)
    print(f'\n📋 JANELAS DESLIZANTES ({janelas_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')

if dummy_reg:
    df_exemplo = pd.read_csv(dummy_reg[0], index_col=0, nrows=5)
    print(f'\n📋 VARIÁVEL DUMMY ({dummy_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')

if indicadores_reg:
    df_exemplo = pd.read_csv(indicadores_reg[0], index_col=0, nrows=5)
    print(f'\n📋 INDICADORES TÉCNICOS ({indicadores_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')

COLUNAS ADICIONADAS EM CADA VARIAÇÃO

📋 JANELAS DESLIZANTES (AGRO3_tratado.csv):
   Colunas: ['Close', 'High', 'Low', 'Open', 'Volume', 'Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut', 'Open_lag2d', 'Open_lag4d', 'Open_lag6d', 'High_lag2d', 'High_lag4d', 'High_lag6d', 'Low_lag2d', 'Low_lag4d', 'Low_lag6d', 'Close_lag2d', 'Close_lag4d', 'Close_lag6d']



📋 VARIÁVEL DUMMY (AGRO3_tratado.csv):
   Colunas: ['Close', 'High', 'Low', 'Open', 'Volume', 'Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut', 'period_1', 'period_2']

📋 INDICADORES TÉCNICOS (AGRO3_tratado.csv):
   Colunas: ['Close', 'High', 'Low', 'Open', 'Volume', 'Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut', 'OBV', 'FWMA', 'TEMA', 'HLC3', 'BB_upper', 'BB_middle', 'BB_lower']
